# 03 — SAE + MMP counterfactual + HTML report

**Goal:** the interpretability layer (Gen 5). Decompose the Uni-Mol
embedding with a Sparse Autoencoder, scan single-substitution MMP rules,
and write a single-page HTML report.

Read `docs/BACKGROUND.md` §3 Gen 5 sub-threads B (SAE) and C (MMP)
first. SAE training is GPU-friendly but runs on CPU for small N.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

from qsar_tutorial.data import load_coadd_pa
from qsar_tutorial.featurizer import UniMolFeaturizer
from qsar_tutorial.model import build_classifier, fit_with_class_weight
from qsar_tutorial.sae import train_sae, dead_ratio, descriptor_recovery
from qsar_tutorial.counterfactual import scan
from qsar_tutorial.report import ReportPayload, render_report
from qsar_tutorial.shap_layer import explain

ds = load_coadd_pa('../data/processed/coadd_pa.csv')
feat = UniMolFeaturizer()
X, valid = feat.featurize(list(ds.smiles))
X, y = X[valid].astype(np.float32), ds.y[valid]
smis = ds.smiles[valid]
print(X.shape)

### Train the Sparse Autoencoder

In [ ]:
sae = train_sae(X, latent_dim=2048, epochs=80, l1_lambda=1e-3, verbose=False)
print('dead_ratio:', dead_ratio(sae, X))

### Descriptor recovery — does the SAE encode physicochemistry?

In [ ]:
RDKIT = ['MolWt','MolLogP','TPSA','NumHAcceptors','NumHDonors',
         'NumRotatableBonds','NumAromaticRings','FractionCSP3','HeavyAtomCount','RingCount']
desc = np.zeros((len(smis), len(RDKIT)), dtype=np.float32)
for i, smi in enumerate(smis):
    m = Chem.MolFromSmiles(smi)
    if m is None: continue
    for j, name in enumerate(RDKIT):
        try: desc[i, j] = float(getattr(Descriptors, name)(m))
        except Exception: pass

Z = sae.encode(X)
r2 = descriptor_recovery(Z, desc)
print('R^2 per descriptor:', dict(zip(RDKIT, r2.round(3))))
print('median:', float(np.median(r2)))

### MMP counterfactual on inactive parents

In [ ]:
final = build_classifier(n_estimators=300, max_depth=6)
fit_with_class_weight(final, X, y)

def score(smis_in):
    Xs, vs = feat.featurize(list(smis_in))
    out = np.zeros(len(smis_in))
    if vs.any():
        out[vs] = final.predict_proba(Xs[vs])[:, 1]
    return out

inactive = list(smis[y == 0][:200])
rules = scan(inactive, score)
for r in rules:
    print(f'{r.name:>32}  n={r.n_applied:4d}  d_p={r.delta_p_mean:+.3f}  CI={r.delta_p_ci95}')

### Render the HTML report

In [ ]:
sv = explain(final, X[:500], feature_names=[f'unimol_{i}' for i in range(X.shape[1])])
payload = ReportPayload(
    title='CO-ADD PA — Uni-Mol pipeline',
    dataset_name='CO-ADD P. aeruginosa (public)',
    n_total=len(y),
    metrics={'AUC': 0.0, 'AUPRC': 0.0, 'MCC@0.5': 0.0},  # fill from OOF earlier
    top_shap=sv.top_features(20),
    sae=dict(latent_dim=2048, dead_ratio=float(dead_ratio(sae, X)),
             r2_median=float(np.median(r2)), r2_above_05=int((r2>0.5).sum()), r2_total=len(r2)),
    mmp_rows=[dict(name=r.name, smirks=r.smirks, n_applied=r.n_applied,
                   delta_p_mean=r.delta_p_mean, delta_p_ci95=r.delta_p_ci95,
                   exemplar_delta=r.exemplar_delta) for r in rules],
)
render_report(payload, '../reports/coadd_pa_report.html')

### What to take away

1. **dead_ratio < 0.05** → SAE healthy.
2. **descriptor R² median > 0.5** → SAE features encode known
   physicochemistry; safe to interpret.
3. **MMP rules**: read the **CI** first, the mean second. Treat any rule
   as *hypothesis-generating*. The parent project required a 95% CI
   excluding 0 *and* a per-parent permutation test before even calling
   something a candidate. See `docs/BACKGROUND.md` §5.3.